In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_directory = 'rca100/cases/t017'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "inventory错误次数突增" (inventory error count surge). It occurred between 2026-03-30 18:40:43 and 2026-03-30 18:44:01 (UTC+8). The alert entity is the operation `inventory::/api/v1/inventory/{productId}` of type `apm.operation` in the `apm` domain. The user is requesting an analysis of the root cause for this issue.

The original code execution output of IPython Kernel is also provided below for reference:

('inventory错误次数突增', {'start': '2026-03-30T18:40:43.549683+08:00', 'end': '2026-03-30T18:44:01+08:00'}, {'entity_id': '162799adab6799326676266a2c96cf3e', 'entity_name': 'inventory::/api/v1/inventory/{productId}', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">inventory错误次数突增<alert_event event_id="0dde689e8b6648f8cd6622a4ee419786" trans_id="teocb7gcld15n8e4bgn1805qa0" rule_id="663b6c72-c01c-49e2-b1a7-805ae65925b3" rule_name="inventory错误次数突增" /></code> 帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# List all available KPIs and their corresponding entity_set values
kpi_entity_set = metrics_data[['metric', 'entity_set']].drop_duplicates().reset_index(drop=True)
kpi_entity_set

```
Out[2]:
```
The `metrics.parquet` file contains 63 unique KPIs across various `entity_set` values. Relevant `entity_set` values for the `inventory` service include `apm.operation` and `apm.service.legacy`, which track metrics like `error_rate`, `error`, `slow`, and `workload`. Other `entity_set` values, such as `k8s.node`, provide node-level metrics like `node_memory_usage_rate` and `node_disk_usage_rate`. These KPIs can help analyze both service-level and node-level performance issues.

The original code execution output of IPython Kernel is also provided below for reference:

metric          entity_set
0        node_ready_status            k8s.node
1    node_not_ready_status            k8s.node
2     node_disk_usage_rate            k8s.node
3   node_memory_usage_rate            k8s.node
4   node_pod_running_count            k8s.node
..                     ...                 ...
58                workload  apm.service.legacy
59                   error  apm.service.legacy
60                    slow  apm.service.legacy
61              error_rate       apm.operation
62              error_rate        apm.instance

[63 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Ensure 'time' and 'value' columns are numeric for calculations
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')

# Create a new column to classify by 'component-KPI'
metrics_data['component_kpi'] = metrics_data['entity_name'] + '-' + metrics_data['metric']

# Calculate the global P95 threshold for each 'component-KPI' time series
p95_thresholds = metrics_data.groupby('component_kpi')['value'].quantile(0.95).reset_index()
p95_thresholds.columns = ['component_kpi', 'p95_threshold']

p95_thresholds

```
Out[3]:
```
The global P95 thresholds have been calculated for 302 unique 'component-KPI' time series across the entire `metrics.parquet` file. These thresholds represent the 95th percentile of metric values for each component-KPI combination, providing a baseline to identify anomalies during the alert time window. The results include various components and KPIs, such as `shipping-workload` and `apiserver_requests_per_second`.

The original code execution output of IPython Kernel is also provided below for reference:

component_kpi  p95_threshold
0                    -apiserver_concurrent_connections       5.000000
1                  -apiserver_non_2xx_read_request_qps       0.088102
2                 -apiserver_non_2xx_write_request_qps       0.088881
3                       -apiserver_requests_per_second      27.008088
4                           -cluster_failed_node_count       0.000000
..                                                 ...            ...
297                                  shipping-workload   12256.500000
298  shipping::oteldemo.ShippingService/GetQuote-av...       0.003729
299  shipping::oteldemo.ShippingService/GetQuote-re...    6130.000000
300  shipping::oteldemo.ShippingService/ShipOrder-a...       0.000011
301  shipping::oteldemo.ShippingService/ShipOrder-r...    6129.000000

[302 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pytz
from datetime import datetime

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat('2026-03-30T18:40:43.549683+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-03-30T18:44:01+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert the alert time window to Unix microseconds for filtering
alert_start_us = int(alert_start.timestamp() * 1_000_000)
alert_end_us = int(alert_end.timestamp() * 1_000_000)

# Filter metrics data within the alert time window
filtered_metrics = metrics_data[(metrics_data['time'] >= alert_start_us) & (metrics_data['time'] <= alert_end_us)]

# Merge the filtered metrics with the global P95 thresholds
merged_data = filtered_metrics.merge(p95_thresholds, on='component_kpi', how='left')

# Identify anomalies where the metric value exceeds the P95 threshold
anomalies = merged_data[merged_data['value'] > merged_data['p95_threshold']]

anomalies

```
Out[4]:
```
During the alert time window, 294 anomalies were identified where metric values exceeded their respective global P95 thresholds. These anomalies span various components and KPIs, including `node_disk_usage_rate` for Kubernetes nodes and `error_rate` for services like `inventory`, `cart`, `frontend`, and `frontend-proxy`. These anomalies indicate potential performance or error issues in these components during the alert period.

The original code execution output of IPython Kernel is also provided below for reference:

time domain    entity_set                         entity_id                      entity_name  ...      value                                     metric_set_id service                               component_kpi p95_threshold
252   1774867273000000    k8s      k8s.node                                                                     ...  33.088612  k8s@metric_set@k8s.metric.high_level_metric_node    None                       -node_disk_usage_rate     32.881330
253   1774867303000000    k8s      k8s.node                                                                     ...  33.090842  k8s@metric_set@k8s.metric.high_level_metric_node    None                       -node_disk_usage_rate     32.881330
254   1774867333000000    k8s      k8s.node                                                                     ...  33.093587  k8s@metric_set@k8s.metric.high_level_metric_node    None                       -node_disk_usage_rate     32.881330
255   1774867363000000    k8s      k8s.node                                                                     ...  33.095852  k8s@metric_set@k8s.metric.high_level_metric_node    None                       -node_disk_usage_rate     32.881330
256   1774867393000000    k8s      k8s.node                                                                     ...  33.097666  k8s@metric_set@k8s.metric.high_level_metric_node    None                       -node_disk_usage_rate     32.881330
...                ...    ...           ...                               ...                              ...  ...        ...                                               ...     ...                                         ...           ...
3979  1774867423000000    apm  apm.instance  fa548039d9a7b76db9ec70507e734faf  frontend-proxy-67d64dc4f8-8c99b  ...   0.171650            apm@metric_set@apm.metric.apm.instance    None  frontend-proxy-67d64dc4f8-8c99b-error_rate      0.170726
3988  1774867333000000    apm  apm.instance  8709d171a640d6264396ee9b86b3604c            cart-67f957cc88-q9w2g  ...   0.761450            apm@metric_set@apm.metric.apm.instance    None            cart-67f957cc88-q9w2g-error_rate      0.761187
4009  1774867423000000    apm  apm.instance  4122d23aef06a1955458774646955bd6                        inventory  ...   0.992453            apm@metric_set@apm.metric.apm.instance    None                        inventory-error_rate      0.991821
4015  1774867423000000    apm  apm.instance  eab531e9b188fe065edbebdbf839c609  frontend-proxy-67d64dc4f8-4ptg6  ...   0.160294            apm@metric_set@apm.metric.apm.instance    None  frontend-proxy-67d64dc4f8-4ptg6-error_rate      0.159924
4018  1774867333000000    apm  apm.instance  beecfe3febef4ed2425bac5626962ed9        frontend-6df7d557f5-nz7g4  ...   0.175999            apm@metric_set@apm.metric.apm.instance    None        frontend-6df7d557f5-nz7g4-error_rate      0.174562

[294 rows x 11 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Sort anomalies by component_kpi and time for easier fault identification
anomalies = anomalies.sort_values(by=['component_kpi', 'time'])

# Group anomalies by component_kpi to analyze each time series separately
grouped_anomalies = anomalies.groupby('component_kpi')

# Initialize a list to store identified faults
faults = []

# Analyze each group for consecutive anomalies
for component_kpi, group in grouped_anomalies:
    # Sort the group by time
    group = group.sort_values(by='time')
    
    # Identify consecutive anomalies by checking time differences
    group['time_diff'] = group['time'].diff().fillna(0)
    
    # Define a threshold for consecutive anomalies (e.g., 1 minute in microseconds)
    consecutive_threshold = 60 * 1_000_000  # 1 minute in microseconds
    
    # Identify start and end of consecutive anomalies
    group['is_consecutive'] = group['time_diff'] <= consecutive_threshold
    group['fault_group'] = (group['is_consecutive'] == False).cumsum()
    
    # Group by fault_group to find time ranges of faults
    for fault_group, fault_data in group.groupby('fault_group'):
        if len(fault_data) > 1:  # Only consider faults with more than one consecutive anomaly
            fault_start = fault_data['time'].min()
            fault_end = fault_data['time'].max()
            faults.append({
                'component_kpi': component_kpi,
                'fault_start': fault_start,
                'fault_end': fault_end
            })

# Convert the faults list to a DataFrame for better readability
faults_df = pd.DataFrame(faults)

# Convert fault_start and fault_end to human-readable timestamps
faults_df['fault_start'] = pd.to_datetime(faults_df['fault_start'], unit='us', utc=True).dt.tz_convert('Asia/Shanghai')
faults_df['fault_end'] = pd.to_datetime(faults_df['fault_end'], unit='us', utc=True).dt.tz_convert('Asia/Shanghai')

faults_df

```
Out[5]:
```
The analysis identified 62 faults where metric values significantly deviated from the global P95 thresholds during the alert time window. Each fault is associated with a specific `component-KPI` and spans a time range. Examples include high `deployment_cpu_usage_total` and `deployment_memory_usage_total` between 18:41:13 and 18:43:43, elevated `node_cpu_usage_rate` between 18:42:13 and 18:43:43, and anomalies in the `recommendation` service metrics like `latency` and `request_count` between 18:42:43 and 18:43:43. These faults highlight potential performance issues in deployments, nodes, and services during the incident.

The original code execution output of IPython Kernel is also provided below for reference:

component_kpi               fault_start                 fault_end
0                         -deployment_cpu_usage_total 2026-03-30 18:41:13+08:00 2026-03-30 18:43:43+08:00
1                     -deployment_cpu_usage_vs_limits 2026-03-30 18:41:13+08:00 2026-03-30 18:43:43+08:00
2                   -deployment_cpu_usage_vs_requests 2026-03-30 18:41:13+08:00 2026-03-30 18:43:43+08:00
3                      -deployment_memory_usage_total 2026-03-30 18:41:13+08:00 2026-03-30 18:43:43+08:00
4                                -node_cpu_usage_rate 2026-03-30 18:42:13+08:00 2026-03-30 18:43:43+08:00
..                                                ...                       ...                       ...
57      recommendation-84c94f9655-hk7wb-request_count 2026-03-30 18:42:43+08:00 2026-03-30 18:43:43+08:00
58                             recommendation-latency 2026-03-30 18:42:43+08:00 2026-03-30 18:43:43+08:00
59                                 recommendation-mem 2026-03-30 18:42:13+08:00 2026-03-30 18:43:43+08:00
60                                recommendation-slow 2026-03-30 18:42:43+08:00 2026-03-30 18:43:13+08:00
61  recommendation::/oteldemo.RecommendationServic... 2026-03-30 18:42:43+08:00 2026-03-30 18:43:43+08:00

[62 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Calculate the magnitude of deviation for each anomaly
anomalies['deviation'] = anomalies['value'] - anomalies['p95_threshold']

# Find the anomaly with the maximum deviation
most_significant_anomaly = anomalies.loc[anomalies['deviation'].idxmax()]

# Extract relevant details for the most significant fault
most_significant_fault = {
    'component_kpi': most_significant_anomaly['component_kpi'],
    'fault_time': pd.to_datetime(most_significant_anomaly['time'], unit='us', utc=True).tz_convert('Asia/Shanghai'),
    'deviation': most_significant_anomaly['deviation']
}

most_significant_fault

```
Out[6]:
```
The most significant fault is associated with the `-deployment_memory_usage_total` KPI. It occurred at 2026-03-30 18:43:43 (UTC+8) with a deviation of approximately 85,018,214.4 from the global P95 threshold, indicating a substantial memory usage issue in the deployment.

The original code execution output of IPython Kernel is also provided below for reference:

{'component_kpi': '-deployment_memory_usage_total', 'fault_time': Timestamp('2026-03-30 18:43:43+0800', tz='Asia/Shanghai'), 'deviation': np.float64(85018214.4000001)}```
```

In [ ]:
# In[7]:

# Load the traces.parquet file
traces_file_path = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Ensure 'startTime' and 'endTime' columns are numeric for filtering
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')

# Filter traces within the alert time window
filtered_traces = traces_data[
    (traces_data['startTime'] >= alert_start_us * 1_000) & 
    (traces_data['endTime'] <= alert_end_us * 1_000)
]

# Focus on services with anomalies identified in metrics analysis
anomalous_services = ['inventory', 'cart', 'frontend', 'frontend-proxy']
filtered_traces_anomalous = filtered_traces[filtered_traces['serviceName'].isin(anomalous_services)]

# Analyze call chains to identify the most downstream faulty service
downstream_faults = filtered_traces_anomalous.groupby('serviceName')['duration'].sum().reset_index()
downstream_faults = downstream_faults.sort_values(by='duration', ascending=False)

downstream_faults

```
Out[7]:
```
The most downstream faulty service identified in the traces is `inventory`, which has the highest cumulative duration of trace spans among the anomalous services. This indicates that `inventory` is likely the primary bottleneck or source of the issue during the alert time window. Other services like `cart`, `frontend-proxy`, and `frontend` also show elevated durations but to a lesser extent.

The original code execution output of IPython Kernel is also provided below for reference:

serviceName                                           duration
3       inventory  3980903685862276407420713175338523489342494751...
0            cart  3303500175210015845002550500568900018390002348...
2  frontend-proxy  3017000268700035600003625000285300025880002659...
1        frontend  2242916255422311186241412070108678513895771534...```
```

In [ ]:
# In[8]:

# Load the logs.parquet file
logs_file_path = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Ensure '_time_' column is in a consistent datetime format
logs_data['_time_'] = pd.to_datetime(logs_data['_time_'], errors='coerce')

# Filter logs for the 'inventory' service during the alert time window
inventory_logs = logs_data[
    (logs_data['_time_'] >= alert_start) & 
    (logs_data['_time_'] <= alert_end) & 
    (logs_data['_container_name_'].str.contains('inventory', case=False, na=False))
]

# Search for error messages or exceptions in the logs
error_logs = inventory_logs[inventory_logs['content'].str.contains('error|exception|fail|critical', case=False, na=False)]

error_logs

```
Out[8]:
```
No error messages, exceptions, or relevant failure logs were found for the `inventory` service during the alert time window. This suggests that the elevated error rate observed in the metrics may not have been directly logged as application-level errors, and the root cause might lie elsewhere, such as in upstream dependencies, configuration issues, or external factors.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _namespace_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```